In [8]:
from pipelines.readers.bcb_historico_taxas_juros import ReaderSQLBCBHistoricoTaxasJuros
import pandas as pd
import plotly.express as px

In [5]:
# Ler os dados históricos de taxas de juros do Banco Central do Brasil (BCB) usando o leitor SQL.
reader = ReaderSQLBCBHistoricoTaxasJuros()
df = reader.read(start_date=None, end_date=None)
df

,Reunião,Data da Reunião,Data da Divulgação,Período de Vigência,Taxa Selic Meta (%),Taxa Selic Over (%),Taxa Selic Meta Anterior (%),Taxa Selic Over Anterior (%),Início da Vigência,Fim da Vigência
0,1ª,1996-06-26,NaT,01/07/1996 - 31/07/1996,1.90,NaN,1.93,23.28,1996-07-01,1996-07-31
1,2ª,1996-07-30,NaT,01/08/1996 - 31/08/1996,1.90,NaN,1.97,25.01,1996-08-01,1996-08-31
2,3ª,1996-08-21,NaT,01/09/1996 - 30/09/1996,1.88,NaN,1.90,25.40,1996-09-01,1996-09-30
3,4ª,1996-09-23,NaT,01/10/1996 - 31/10/1996,1.82,1.93,1.86,23.48,1996-10-01,1996-10-31
4,5ª,1996-10-23,NaT,01/11/1996 - 30/11/1996,1.78,1.90,1.80,25.27,1996-11-01,1996-11-30
...,...,...,...,...,...,...,...,...,...,...
282,275ª,2025-12-10,NaT,11/12/2025 - 28/01/2026,15.00,NaN,1.84,14.90,2025-12-11,2026-01-28
283,276ª,2026-01-28,NaT,29/01/2026 - 18/03/2026,15.00,NaN,1.84,14.90,2026-01-29,2026-03-18
284,277ª,2026-03-18,NaT,19/03/2026 - 29/04/2026,14.75,NaN,1.53,14.65,2026-03-19,2026-04-29
285,278ª,2026-04-29,NaT,30/04/2026 - 17/06/2026,14.50,NaN,1.78,14.40,2026-04-30,2026-06-17


In [ ]:
# Gerar um gráfico de linha da evolução da Taxa Selic Meta (%) ao longo do tempo.
df_plot = df.copy()
df_plot["Data da Reunião"] = pd.to_datetime(df_plot["Data da Reunião"], errors="coerce")
df_plot = df_plot.sort_values("Data da Reunião")

fig = px.line(
    df_plot,
    x="Data da Reunião",
    y="Taxa Selic Meta (%)",
    title="Evolução da Taxa Selic Meta (%)",
    markers=True,
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Data da Reunião",
    yaxis_title="Taxa Selic Meta (%)",
)

fig.show()

In [10]:
# Distribuicao da Taxa Selic Meta (%) com destaque para o valor atual
df_dist = df.copy()
df_dist["Data da Reunião"] = pd.to_datetime(df_dist["Data da Reunião"], errors="coerce")
df_dist["Taxa Selic Meta (%)"] = pd.to_numeric(df_dist["Taxa Selic Meta (%)"], errors="coerce")
df_dist = df_dist.dropna(subset=["Data da Reunião", "Taxa Selic Meta (%)"])

# Valor atual = ultima observacao disponivel no tempo
current_idx = df_dist["Data da Reunião"].idxmax()
current_value = float(df_dist.loc[current_idx, "Taxa Selic Meta (%)"])
current_date = df_dist.loc[current_idx, "Data da Reunião"]

fig_dist = px.histogram(
    df_dist,
    x="Taxa Selic Meta (%)",
    nbins=30,
    marginal="box",
    title="Distribuicao da Taxa Selic Meta (%)",
    template="plotly_white",
)

# Marca o ponto atual na distribuicao
fig_dist.add_vline(
    x=current_value,
    line_width=2,
    line_dash="dash",
    line_color="crimson",
)

fig_dist.add_annotation(
    x=current_value,
    y=1.02,
    yref="paper",
    showarrow=False,
    text=f"Atual: {current_value:.2f}% ({current_date.strftime('%d/%m/%Y')})",
    font=dict(color="crimson"),
)

fig_dist.update_layout(
    xaxis_title="Taxa Selic Meta (%)",
    yaxis_title="Frequencia",
)

fig_dist.show()